# basic

In [1]:
%load_ext autoreload
%autoreload all

In [ ]:
import polars as pl
import numpy as np
import pickle
import itertools

import configs_mapped as configs_mapped
from src.tokenizer import GraphTokenizer
from src.iterative_selection import iterative_approach_w_graph_red, IterativeMarginalGainMax

# 0. load files

In [ ]:
with open(configs_mapped.ProcessedGraph().candidate_is_a_reachable_dict, "rb") as f:
    candidate_reachable_child_map = pickle.load(f)

df_concept_hug_w_depth = pl.read_parquet(configs_mapped.ProcessedGraph.concept_w_depth)
mapped_candidate_rel_dist_prop = (pl.read_parquet(configs_mapped.ProcessedGraph().mapped_candidate_rel_dist_prop)
                                  
                                  # remove all relations in subgraph where candidate and mapped are distant
                                  .filter(pl.col("distance") <= configs_mapped.TokenizerParam().max_dist_candidate)
                                  )

In [ ]:
pl.read_parquet(configs_mapped.ProcessedGraph.concept_w_depth)


id,label,concept_type,sem_tag,top_category,collection,is_mapped,max_depth,min_depth
str,str,str,str,str,list[str],bool,i64,i64
"""127265005""","""Neoplasm of popliteal lymph no…","""SCT_PRE""","""disorder""","""finding""",null,false,10,5
"""273547007""","""Katz activities of daily livin…","""SCT_PRE""","""assessment scale""","""staging scale""",null,false,4,4
"""276134001""","""Second value (attribute)""","""SCT_PRE""","""attribute""","""metadata""",null,false,6,6
"""117033003:116686009=(168139001…","""117033003|Aerobic microbial cu…","""SCT_POST""","""procedure""","""procedure""","[""labo""]",true,8,7
"""1269322006""","""Insertion of denture (procedur…","""SCT_PRE""","""procedure""","""procedure""",null,false,7,3
…,…,…,…,…,…,…,…,…
"""41373000""","""Entire sacral venous plexus (b…","""SCT_PRE""","""body structure""","""body structure""",null,false,16,10
"""332247009""","""Product containing precisely d…","""SCT_PRE""","""clinical drug""","""product""",null,false,7,6
"""419495008""","""Percentage inhibition (qualifi…","""SCT_PRE""","""qualifier value""","""qualifier value""",null,false,5,5


# 1. iterative with graph reduction

In [ ]:
distance_score_types = ["sum", "tempered_sum"]
distance_type = ["inv", "inv_exp"]

for sc_type, d_type in itertools.product(distance_score_types, distance_type):
    candidate_rows = iterative_approach_w_graph_red(
        mapped_candidate_rel_dist_prop,
        candidate_reachable_child_map,
        sc_type,
        d_type,
    )
    pl.DataFrame(candidate_rows).write_parquet(f"{configs_mapped.IterativeGraphRed().path} {sc_type}_{d_type}.parquet")
    print(f"{sc_type} / {d_type}: {len(candidate_rows)} candidates selected")

number of candidates selected: 500, number of rows remaining: 417221
number of candidates selected: 1000, number of rows remaining: 329179
number of candidates selected: 1500, number of rows remaining: 288759
number of candidates selected: 2000, number of rows remaining: 273790
number of candidates selected: 2500, number of rows remaining: 273290
number of candidates selected: 3000, number of rows remaining: 272790
number of candidates selected: 3500, number of rows remaining: 272290
number of candidates selected: 4000, number of rows remaining: 271790
number of candidates selected: 4500, number of rows remaining: 271290
number of candidates selected: 5000, number of rows remaining: 270790
number of candidates selected: 5500, number of rows remaining: 270290
number of candidates selected: 6000, number of rows remaining: 269790
number of candidates selected: 6500, number of rows remaining: 269290
number of candidates selected: 7000, number of rows remaining: 268790
number of candidates 

# 2. marginal gain

In [ ]:
distance_types = ["inv", "inv_exp"]

max_candidate = mapped_candidate_rel_dist_prop["src.id"].n_unique()

for d_type in distance_types:
    margin_gain_iterative = IterativeMarginalGainMax(mapped_candidate_rel_dist_prop, d_type, max_candidates=max_candidate)
    selected_marginal_gain, _ = margin_gain_iterative.iterative_marginal_gain_greedy_without_k()
    selected_marginal_gain.write_parquet(f"{configs_mapped.IterativeMarginalGain().path}_{d_type}.parquet")


Marginal gain greedy: 1000it [01:35, 12.01it/s]

selected=1000, best_gain=7.9167, gain_ratio=0.0010, mean_state_score=0.4154


Marginal gain greedy: 2000it [02:54, 13.14it/s]

selected=2000, best_gain=3.2500, gain_ratio=0.0004, mean_state_score=0.4464


Marginal gain greedy: 3000it [04:07, 14.00it/s]

...

selected=45000, best_gain=0.6321, gain_ratio=0.0001, mean_state_score=0.5231


Marginal gain greedy: 46000it [46:36, 19.05it/s]

selected=46000, best_gain=0.6321, gain_ratio=0.0001, mean_state_score=0.5270


Marginal gain greedy: 46150it [46:44, 16.46it/s]

Reached max_candidates=46150.


# tokenizer test (max dist  == 3)

In [ ]:
tokenizer = GraphTokenizer(df_concept_hug_w_depth,
                        mapped_candidate_rel_dist_prop,
                        candidate_reachable_child_map,
                        configs_mapped.TokenizerParam().max_dist_candidate
                        )

## tryout

In [6]:
# top 8000 candidates of highest degree
for k in np.arange(1000, 10000, 1000):
    example_candidates = mapped_candidate_rel_dist_prop.group_by("dst.id").agg(num_src = pl.col("src.id").n_unique()).sort(by = "num_src", descending=True).head(k)["dst.id"].to_list()
    scores, results, df_tok_all_n_dist = tokenizer.evaluate_components_and_tokenize(example_candidates, debug=True)


In [7]:
df_tok_all_n_dist# ["candidate_id"].n_unique()

mapped_id,candidate_id,relation,distance
str,str,str,f64
"""73862001""","""73862001""","""IS_A""",0.0
"""254502002""","""254502002""","""IS_A""",0.0
"""127337006""","""127337006""","""IS_A""",0.0
"""414403008""","""414403008""","""IS_A""",0.0
"""88425004""","""88425004""","""IS_A""",0.0
…,…,…,…
"""277387000""","""UNK""","""IS_A""",null
"""1296723006""","""UNK""","""IS_A""",null
"""10083006""","""UNK""","""IS_A""",null
